# Create Persian dataset
After `scripts/scrape.py` and `scripts/filter_persian.py` are run, this notebook builds `train/` and `eval/` in `data/persian/` from `data/persian/raw/`. It filters images from `data/persian/raw/` via `scripts/filter_persian.py`, captions them via `scripts/caption.py`, and splits them into train and eval via random sampling.

## Environment/setup

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

fatal: destination path 'stable-diffusion-rebuilt' already exists and is not an empty directory.
/content/stable-diffusion-rebuilt
Already up to date.


In [3]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


In [4]:
# mount drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
# symlink colab data/persian/ -> drive data/persian
!mkdir -p data # create colab data/
!ln -sfn /content/drive/MyDrive/sd-rebuilt/data/persian data/persian # colab data/persian points to drive persian

## Filter and caption

In [6]:
# 1) filter raw images
# separates into raw/ and rejected/
!python scripts/filter_persian.py --data-dir data/persian/raw

6843/6843  moved: 1604
done. moved 1604 images to /content/stable-diffusion-rebuilt/data/persian/rejected, kept 5239


In [7]:
# 2) caption images
# adds metadata.jsonl (and metadata_raw.jsonl) to data_dir
!python scripts/caption.py --data-dir data/persian/raw


>>> Generating BLIP captions
image batches: 100% 328/328 [06:44<00:00,  1.23s/it]

>>> Cleaning BLIP captions

>>> Done. Metadata saved to /content/stable-diffusion-rebuilt/data/persian/raw/metadata.jsonl


In [8]:
# 3) create train/eval splits
import json
import random
from pathlib import Path
import shutil

raw_dir = Path("data/persian/raw")
train_size = 1300
eval_size = 165
seed = 42

# sample TRAIN_SIZE + EVAL_SIZE images from raw/, then split
paths = sorted((raw_dir / "images").glob("*.jpg"))
sample = random.Random(seed).sample(paths, train_size + eval_size)
splits = {"train": sample[:train_size], "eval": sample[train_size:]}

# get captions by file_name
captions = {}
with open(raw_dir / "metadata.jsonl") as meta:
    for line in meta:
        row = json.loads(line)
        captions[row["file_name"]] = row["text"]

for split, split_files in splits.items():
    out_dir = raw_dir.parent / split # data/persian/<split>/
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    with open(out_dir / "metadata.jsonl", "w") as meta:
        for p in split_files:
            shutil.copy(p, out_dir / "images" / p.name)
            fn = f"images/{p.name}"
            meta.write(json.dumps({"file_name": fn, "text": captions[fn]}) + "\n")

In [ ]:
# 4) push raw/ to HF ??